# Ronda_15 - Geracao em Massa das Classes Piores

Este notebook corre a geracao massiva apenas para os datasets `pt/1159_7` e `pt/9338`, avalia todas as imagens por metricas (CLIP, LPIPS, RMSE, SSIM e metricas visuais), e guarda as 20 melhores por categoria.


In [6]:
from __future__ import annotations

import gc
import json
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import torch
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "tp2_common.py").exists():
            return p
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tp2_common import LCMConfig, EvalConfig, load_image, load_lcm_pipeline, render_prompt, seed_from_filename, write_csv
from tp2_metrics import (
    add_weighted_scores,
    clip_image_embedding,
    load_clip,
    load_lpips,
    pil_to_lpips_tensor,
    pixel_metrics_with_ssim,
    target_region_metrics,
    visual_specific_metrics,
)

OUTPUTS_ROOT = PROJECT_ROOT / "TP2-students" / "students" / "outputs"
ronda_15_dir = OUTPUTS_ROOT / "ronda_15_gerracaoclasses_piores"
ronda_15_dir.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = ronda_15_dir / "datasets"
TARGET_ROOT = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
RUN_ROOT = ronda_15_dir / "runs_from_dataset"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# usar cache local controlada pelo projeto
HF_CACHE = PROJECT_ROOT / ".hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)

PREFERRED_RENDER_DEVICE = "cuda"
PREFERRED_METRIC_DEVICE = "cuda"
OFFLINE_MODE = False   # <- importante: False para permitir download do modelo
TOP_K = 20
PROMPT_LIMIT = None  # ex.: 300 para testes rapidos


def resolve_device(preferred: str) -> str:
    if preferred == "cuda" and torch.cuda.is_available():
        return "cuda"
    if preferred == "mps" and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    if preferred == "cpu":
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


RENDER_DEVICE = resolve_device(PREFERRED_RENDER_DEVICE)
METRIC_DEVICE = resolve_device(PREFERRED_METRIC_DEVICE)

if PREFERRED_RENDER_DEVICE == "cuda" and RENDER_DEVICE != "cuda":
    print(f"[Aviso] CUDA nao disponivel para render. A usar: {RENDER_DEVICE}")
if PREFERRED_METRIC_DEVICE == "cuda" and METRIC_DEVICE != "cuda":
    print(f"[Aviso] CUDA nao disponivel para metricas. A usar: {METRIC_DEVICE}")

if OFFLINE_MODE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Ronda_15 dir:", ronda_15_dir)
print("DATASET_ROOT:", DATASET_ROOT)
print("TARGET_ROOT:", TARGET_ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("HF_HOME:", os.environ["HF_HOME"])
print("Render device:", RENDER_DEVICE)
print("Metric device:", METRIC_DEVICE)
print("OFFLINE_MODE:", OFFLINE_MODE)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

PROJECT_ROOT: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2
Ronda_15 dir: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\outputs\ronda_15_gerracaoclasses_piores
DATASET_ROOT: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\outputs\ronda_15_gerracaoclasses_piores\datasets
TARGET_ROOT: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\tp2-chosen
RUN_ROOT: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\outputs\ronda_15_gerracaoclasses_piores\runs_from_dataset
HF_HOME: C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\.hf_cache
Render device: cuda
Metric device: cuda
OFFLINE_MODE: False


## Dataset PT - Como foi gerado

O dataset PT foi construido com a mesma logica da ronda_14: blocos de variacao semantica, produto cartesiano e ordenacao por distancia ao bloco-base. Nesta ronda ha 10 000 prompts para `1159_7` e 10 000 prompts para `9338`, com blocos mais especificos para aproximar melhor os targets.


In [7]:
_MODEL_CACHE: dict[tuple[str, str, bool], dict[str, Any]] = {}


def get_models(render_device: str, metric_device: str, offline_mode: bool) -> dict[str, Any]:
    key = (render_device, metric_device, offline_mode)
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]

    lcm_cfg = LCMConfig()
    eval_cfg = EvalConfig()
    pipe, resolved_render_device = load_lcm_pipeline(lcm_cfg, render_device)
    if hasattr(pipe, "set_progress_bar_config"):
        pipe.set_progress_bar_config(disable=True)

    clip_processor, clip_model = load_clip(eval_cfg.clip_model, metric_device, local_files_only=offline_mode)
    lpips_model = load_lpips(eval_cfg.lpips_net, metric_device)

    bundle = {
        "lcm_cfg": lcm_cfg,
        "eval_cfg": eval_cfg,
        "pipe": pipe,
        "resolved_render_device": resolved_render_device,
        "clip_processor": clip_processor,
        "clip_model": clip_model,
        "lpips_model": lpips_model,
    }
    _MODEL_CACHE[key] = bundle
    return bundle


def load_dataset(language: str, category: str) -> dict[str, Any]:
    language_dir = "pt" if language == "pt" else "chines"
    dataset_path = DATASET_ROOT / language_dir / f"{category}.json"
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset nao encontrado: {dataset_path}")
    return json.loads(dataset_path.read_text(encoding="utf-8"))


def evaluate_prompt(
    *,
    target_name: str,
    target_image,
    target_clip,
    target_lpips,
    prompt_entry: dict[str, Any],
    seed: int,
    models: dict[str, Any],
    metric_device: str,
) -> dict[str, Any]:
    image = render_prompt(
        prompt=prompt_entry["prompt"],
        seed=seed,
        pipe=models["pipe"],
        config=models["lcm_cfg"],
        device=models["resolved_render_device"],
    )

    if image.size != target_image.size:
        image = image.resize(target_image.size)

    render_clip = clip_image_embedding(image, models["clip_processor"], models["clip_model"], metric_device)
    clip_iisim = float((target_clip * render_clip).sum().item())
    render_lpips = pil_to_lpips_tensor(image, metric_device)
    with torch.no_grad():
        lpips_alex = float(models["lpips_model"](target_lpips, render_lpips).item())

    pixel_mse, pixel_rmse, pixel_ssim = pixel_metrics_with_ssim(target_image, image)
    visual = visual_specific_metrics(target_image, image)
    region = target_region_metrics(target_name, target_image, image)

    return {
        "target_name": target_name,
        "id": prompt_entry["id"],
        "prompt": prompt_entry["prompt"],
        "seed_index": prompt_entry.get("seed_index", ""),
        "combo": str(prompt_entry.get("combo", "")),
        "clip_iisim": clip_iisim,
        "lpips_alex": lpips_alex,
        "pixel_mse_01": pixel_mse,
        "pixel_rmse_01": pixel_rmse,
        "pixel_ssim_01": pixel_ssim,
        **visual,
        **region,
    }


def run_category(
    *,
    language: str,
    category: str,
    prompt_limit: int | None = None,
    top_k: int = 20,
) -> Path:
    payload = load_dataset(language, category)
    target_name = payload["target_name"]
    target_path = TARGET_ROOT / target_name
    if not target_path.exists():
        raise FileNotFoundError(f"Target nao encontrado: {target_path}")

    prompts = payload["prompts"]
    if prompt_limit is not None:
        prompts = prompts[:prompt_limit]

    now = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = RUN_ROOT / f"{now}_{language}_{category}"
    run_dir.mkdir(parents=True, exist_ok=False)

    models = get_models(RENDER_DEVICE, METRIC_DEVICE, OFFLINE_MODE)
    seed = seed_from_filename(target_name)

    target_image = load_image(target_path)
    target_clip = clip_image_embedding(target_image, models["clip_processor"], models["clip_model"], METRIC_DEVICE)
    target_lpips = pil_to_lpips_tensor(target_image, METRIC_DEVICE)

    rows: list[dict[str, Any]] = []
    bar = tqdm(prompts, total=len(prompts), desc=f"{language}:{category} avaliacao", dynamic_ncols=True)
    for idx, entry in enumerate(bar, start=1):
        row = evaluate_prompt(
            target_name=target_name,
            target_image=target_image,
            target_clip=target_clip,
            target_lpips=target_lpips,
            prompt_entry=entry,
            seed=seed,
            models=models,
            metric_device=METRIC_DEVICE,
        )
        rows.append(row)

        if idx % 25 == 0:
            bar.set_postfix({"idx": idx, "clip": f"{row['clip_iisim']:.4f}", "lpips": f"{row['lpips_alex']:.4f}"})
        if torch.cuda.is_available() and idx % 200 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    add_weighted_scores(rows)
    rows_sorted = sorted(rows, key=lambda r: float(r["selection_score"]), reverse=True)
    for rank, row in enumerate(rows_sorted, start=1):
        row["rank"] = rank

    write_csv(run_dir / "metrics_all.csv", rows_sorted)

    top_rows = rows_sorted[:top_k]
    top_dir = run_dir / "top20"
    top_dir.mkdir(parents=True, exist_ok=True)

    for rank, row in enumerate(tqdm(top_rows, desc=f"{language}:{category} guardar top20", dynamic_ncols=True), start=1):
        image = render_prompt(
            prompt=row["prompt"],
            seed=seed,
            pipe=models["pipe"],
            config=models["lcm_cfg"],
            device=models["resolved_render_device"],
        )
        image_path = top_dir / f"rank_{rank:02d}_{row['id']}.png"
        image.save(image_path)
        row["top_image_path"] = str(image_path)

    (run_dir / "top20_metrics.json").write_text(
        json.dumps(top_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    summary = {
        "language": language,
        "category": category,
        "target_name": target_name,
        "dataset_prompt_count": payload.get("prompt_count", len(payload["prompts"])),
        "evaluated_prompt_count": len(prompts),
        "render_device": models["resolved_render_device"],
        "metric_device": METRIC_DEVICE,
        "top_k": top_k,
        "best_selection_score": float(top_rows[0]["selection_score"]) if top_rows else None,
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Concluido: {language} {category} -> {run_dir}")
    return run_dir


In [8]:
import time

RUN_SMOKE_TEST = False
N_TESTE = 10
N_TOTAL = 20000

if RUN_SMOKE_TEST:
    t0 = time.perf_counter()
    run_category(language="pt", category="1159_7", prompt_limit=N_TESTE, top_k=20)
    elapsed = time.perf_counter() - t0

    print(f"Tempo para {N_TESTE}: {elapsed:.1f}s")
    print(f"Estimativa para {N_TOTAL}: {elapsed * (N_TOTAL / N_TESTE) / 3600:.2f}h")
else:
    print("Smoke test desativado. Coloca RUN_SMOKE_TEST = True se quiseres estimar tempo.")


Smoke test desativado. Coloca RUN_SMOKE_TEST = True se quiseres estimar tempo.


In [9]:
RUNS_TO_EXECUTE = [
    ("pt", "1159_7"),
    ("pt", "9338"),
]

for language, category in RUNS_TO_EXECUTE:
    run_category(language=language, category=category, prompt_limit=PROMPT_LIMIT, top_k=TOP_K)


Loading pipeline components...: 100%|██████████| 7/7 [00:04<00:00,  1.68it/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\.venv_win\Lib\site-packages\lpips\weights\v0.1\alex.pth


pt:1159_7 guardar top20: 100%|██████████| 20/20 [00:23<00:00,  1.19s/it]


Concluido: pt 1159_7 -> C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\outputs\ronda_15_gerracaoclasses_piores\runs_from_dataset\20260603-092124_pt_1159_7


pt:9338 guardar top20: 100%|██████████| 20/20 [00:23<00:00,  1.19s/it]

Concluido: pt 9338 -> C:\Users\josec\Documents\Programaçao\Projeto2\Projeto2\TP2-students\students\outputs\ronda_15_gerracaoclasses_piores\runs_from_dataset\20260603-133402_pt_9338
